# Expert Knowledge Worker: A First Look at RAG

Building a question-answering assistant for a fictional insurance-tech company,
**Insurellm**. The goal: an assistant that's accurate about company-specific
facts (employees, products) and cheap to run.

This is where **RAG (Retrieval Augmented Generation)** comes in -- instead of
fine-tuning a model on company data (expensive, slow to update), I feed
relevant facts into the prompt at question time, pulled from a knowledge base
of plain text files.

This first version is deliberately the simplest possible approach: brute-force
keyword matching, no embeddings, no vector database. Good for understanding
*why* RAG works before reaching for more sophisticated retrieval later.

**Why RAG is worth learning well:** it's one of the most immediately useful
techniques for applying LLMs to a real business -- nuanced querying across a
company's own documents (contracts, specs, policies) without needing to train
anything. Low cost, quick to stand up, and it directly solves the "the model
doesn't know about our specific business" problem.


## Setup


In [ ]:
import os
import glob
from dotenv import load_dotenv
from pathlib import Path
import gradio as gr
from openai import OpenAI


In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-nano"
openai_client = OpenAI()


## Loading the knowledge base

The "knowledge base" here is just a folder of plain text files -- one per
employee, one per product. Reading employee files first into a dictionary
keyed by surname (pulled out of the filename), so I can look someone up by
name later.


In [ ]:
knowledge = {}

filenames = glob.glob("knowledge-base/employees/*")

for filename in filenames:
    name = Path(filename).stem.split(' ')[-1]
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()


In [ ]:
knowledge


Checking that a specific employee's record loaded correctly.


In [ ]:
knowledge["lancaster"]


Now adding product docs into the same dictionary -- same pattern, different
folder, keyed by product filename instead of a person's surname.


In [ ]:
filenames = glob.glob("knowledge-base/products/*")

for filename in filenames:
    name = Path(filename).stem
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()


In [ ]:
knowledge.keys()


## The system prompt

Setting up the assistant's persona, plus a placeholder at the end where
retrieved context will get appended before each request.


In [ ]:
SYSTEM_PREFIX = """
You represent Insurellm, the Insurance Tech company.
You are an expert in answering questions about Insurellm; its employees and its products.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""


## Retrieval: the brute-force approach

This is the core idea of RAG, done in the simplest way I can think of: strip
punctuation from the user's message, lowercase it, split into words, and check
each word against the `knowledge` dictionary's keys. Any match gets pulled in
as context.


In [ ]:
def get_relevant_context_simple(message):
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    relevant_context = []
    for word in words:
        if word in knowledge:
            relevant_context.append(knowledge[word])
    return relevant_context


## Rewriting it as a list comprehension

Same logic, more compact -- worth doing once I'm sure the loop version is
correct, since it's easier to read at a glance once compressed like this.


In [ ]:
def get_relevant_context(message):
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    return [knowledge[word] for word in words if word in knowledge]


In [ ]:
get_relevant_context("Who is lancaster?")


In [ ]:
get_relevant_context("Who is Lancaster and what is carllm?")


Good sign: asking about two different things (a person and a product) in one
message pulls back both pieces of context, since each word gets checked
independently against the dictionary.


## Formatting retrieved context for the prompt

Turning the list of matched context strings into readable text to append to the
system prompt -- with a clear fallback message when nothing matched, so the
model knows explicitly that no relevant info was found (rather than silently
getting an empty context block).


In [ ]:
def additional_context(message):
    relevant_context = get_relevant_context(message)
    if not relevant_context:
        result = "There is no additional context relevant to the user's question."
    else:
        result = "The following additional context might be relevant in answering the user's question:\n\n"
        result += "\n\n".join(relevant_context)
    return result


In [ ]:
print(additional_context("Who is Alex Lancaster?"))


## Putting it together: the chat function

Same `chat(message, history)` pattern from earlier chatbot notebooks, but now
the system message is built dynamically on every call -- `SYSTEM_PREFIX` plus
whatever context got retrieved for this specific question.


In [ ]:
def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message)
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai_client.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


## Trying it out in Gradio

Same `gr.ChatInterface` pattern as before -- a quick way to prototype and
actually talk to this assistant.


In [ ]:
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)


## What's limited about this approach

Worth being honest about the weaknesses here before moving on to smarter
retrieval:

- **Exact word matching only.** Asking about "Lancaster's role" works, but a
  typo, a nickname, or a different phrasing of the same name might miss
  entirely.
- **No understanding of meaning.** If someone asks "who manages the claims
  team?" without using any name that appears as a dictionary key, nothing gets
  retrieved, even if the answer exists somewhere in the knowledge base.
- **Doesn't scale.** This works because filenames happen to align neatly with
  words people are likely to type. A knowledge base with longer, less
  predictable documents would need actual semantic search -- embeddings and a
  vector database -- to retrieve well.

This version is a genuinely useful baseline, though: it's fast, free (no
embedding costs), and completely transparent about why something did or didn't
get retrieved. Good reason to build this first before adding complexity.
